# Exemplo 10 - Turismo: seu primeiro agente de IA com Google ADK

### Palestra Machine Learning e IA: Da máquina que aprende à IA que age

### Prof. Dr. Ahirton Lopes (https://github.com/ahirtonlopes)

**Onde isso aparece no mercado:** Assistentes de viagem, atendimento ao cliente e copilotos corporativos já combinam LLMs com ferramentas (busca, APIs) e memória para executar tarefas, não só responder perguntas.

**Conceito da palestra:** Um agente é um LLM que decide quais ferramentas usar e age para cumprir um objetivo: é a IA que age.

> Este notebook exige uma API key gratuita do Google AI Studio. O passo a passo para gerar e guardar a chave no Colab está no README do repositório.

# 🚀 Bem-vindo(a) à sua Aventura ADK - Ferramentas & Memória! 🚀

Boas-vindas, Arquiteto(a) de Agentes! Este notebook é o seu guia para dar aos seus agentes de IA dois superpoderes essenciais: ferramentas customizadas e memória de conversa.

Ao final desta aventura, você será capaz de:

- **Construir um Agente Fundamental**: criar um agente de IA simples mas eficaz, do zero, usando o Google Agent Development Kit (ADK).

- **Conceder Novas Habilidades com Ferramentas Customizadas**: ensinar um agente a executar novas tarefas conectando-o a APIs externas, como um serviço de clima em tempo real.

- **Criar um Time de Agentes**: montar um sistema multiagente em que um agente principal delega tarefas especializadas a outros agentes.

- **Dominar a Memória de Conversa**: entender o papel crítico das Sessions para que os agentes lembrem de interações anteriores, lidem com feedback e mantenham uma conversa coerente.

Vamos começar esta aventura!

## Autora Original

Oi, eu sou Qingyue (Annie) Wang, developer advocate e engenheira de IA no **Google**, apaixonada por ajudar pessoas desenvolvedoras a construir com IA e tecnologias de nuvem :)

Se tiver dúvidas sobre o material original, fale com ela no [LinkedIn](https://www.linkedin.com/in/anniewangtech/), [X](https://twitter.com/anniewangtech) ou pelo e-mail anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Bons estudos com Agentes de IA :)
```

> 📝 **Nota desta versão**: o código deste notebook foi modificado em relação ao original - modelos Gemini atualizados, retry automático contra limites de quota, e a ferramenta de clima agora usa a API Open-Meteo com **dados de locais de São Paulo, SP (Brasil)**. Os textos foram traduzidos para PT-BR e os exemplos ambientados em situações típicas paulistanas. Adaptação: [Dr. Ahirton Lopes](https://github.com/ahirtonlopes).


-------------
### 🎁 🛑 Pré-requisito Importante: Prepare o seu Ambiente! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Pegue a sua API Key AQUI**: [Google AI Studio](https://aistudio.google.com/app/apikey)

 -----------------------------------------------------------------------------


```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Parte 0: Setup & Autenticação 🔑

Antes de tudo, vamos deixar as ferramentas prontas. Esta etapa instala as bibliotecas necessárias e configura com segurança a sua API key do Google, para que os agentes possam acessar o poder do Gemini.

In [ ]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")



### Configure a sua API Key
Para usar os modelos Gemini, você precisa de uma API key do [Google AI Studio](https://aistudio.google.com/app/apikey). Esta seção coleta a sua chave com segurança e a configura para o ADK.


In [ ]:
# --- API Key Configuration ---
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the 🔑 icon in the left sidebar, add a secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    # Option 2: Paste it directly (less secure but fine for learning)
    import getpass
    GOOGLE_API_KEY = getpass.getpass("🔑 Enter your Google AI Studio API key: ")
    print("✅ API key entered manually.")


In [ ]:
# --- Set Environment Variables for ADK ---

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

print(f"✅ API key configured (starts with '{GOOGLE_API_KEY[:6]}...')")
print("✅ Using Google AI Studio (not Vertex AI).")


# --- API key sanity check ---
# Google AI Studio currently issues keys in two formats: "AIza..." (classic)
# and "AQ..." (newer). Anything else is probably not an API key (e.g. an OAuth
# token from Cloud Shell / gemini-cli) and will cause 401 UNAUTHENTICATED.
if not GOOGLE_API_KEY or not GOOGLE_API_KEY.startswith(("AIza", "AQ")):
    print("🚨 ATENÇÃO: isso não parece uma API key do AI Studio (formatos atuais: 'AIza...' ou 'AQ...').")
    print("   Gere uma chave em https://aistudio.google.com/app/apikey e atualize o secret GOOGLE_API_KEY.")


In [ ]:
# --- 🔎 Which models are available for this API key? ---
# Lists every Gemini API model and the actions each one supports.
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

print(f"{'MODELO':<50} RECURSOS SUPORTADOS")
print("-" * 100)
for m in client.models.list():
    actions = ", ".join(m.supported_actions or [])
    print(f"{m.name.removeprefix('models/'):<50} {actions}")

# --- 🎯 Modelo usado por todos os agentes deste notebook ---
# NOTE: gemini-2.5-flash was NOT discontinued (official shutdown: Oct/16/2026).
# On Jul/09/2026 a misconfiguration on Google's side made it return 404
# ("no longer available to new users"); the team rolled it back:
# https://discuss.ai.google.dev/t/gemini-2-5-flash-suddenly-return-404/174225
# We still default to gemini-3.1-flash-lite: higher free-tier quota
# (15 RPM vs 5 RPM) and no exposure to the rollback instability.
# Alternativas atuais (jul/2026):
#   - "gemini-3.1-flash-lite" -> geralmente 15 RPM no free tier
#   - "gemini-2.5-flash-lite" -> geralmente 10 RPM, boa reserva se a principal esgotar
#   - "gemini-flash-latest"   -> alias que sempre aponta para o flash mais recente
MODEL_NAME = "gemini-3.1-flash-lite"
print(f"\n✅ Modelo escolhido para os agentes: {MODEL_NAME}")

# --- 🛡️ Automatic retry for 429 errors (bursts of calls) ---
# Each query fires several model calls (router -> agents -> searches).
# With HttpRetryOptions the SDK itself waits and retries on 429
# instead of crashing the whole workflow.
from google.genai import types as genai_types

RETRY_CONFIG = genai_types.GenerateContentConfig(
    http_options=genai_types.HttpOptions(
        retry_options=genai_types.HttpRetryOptions(
            attempts=5,        # up to 5 attempts per call
            initial_delay=10,  # wait 10s before the first retry...
            exp_base=2,        # ...doubling on each retry (10s, 20s, 40s...)
            max_delay=60,
        )
    )
)

PAUSE_BETWEEN_QUERIES = 30  # seconds between test queries (respects the RPM limit)
print(f"🛡️ Retry automático configurado | pausa entre queries: {PAUSE_BETWEEN_QUERIES}s")

---
## Parte 1: Seu Primeiro Agente - O Gênio do Passeio de Um Dia 🧞

Conheça a sua primeira criação! O `day_trip_agent` é um assistente simples, porém poderoso. Vamos deixá-lo um pouco mais esperto ensinando-o a entender **restrições de orçamento**.

* **Agent**: o cérebro da operação, definido por suas instruções, ferramentas e pelo modelo de IA que utiliza.
* **Session**: o histórico da conversa. Para este agente simples, é só um contêiner para um único par pergunta-resposta.
* **Runner**: o motor que conecta o `Agent` e a `Session` para processar o seu pedido e obter a resposta.

```
+--------------------------------------------------+
|      Agente de Passeio Espontâneo de 1 Dia 🤖    |
|--------------------------------------------------|
|  Modelo: gemini-3.1-flash-lite                   |
|  Descrição:                                      |
|   Gera roteiros de um dia inteiro com base em    |
|   humor, interesses e orçamento                  |
|--------------------------------------------------|
|  🔧 Ferramentas:                                 |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capacidades:                                 |
|   - Consciência de orçamento (barato / luxo)     |
|   - Combina com o humor (aventura, relax, etc.)  |
|   - Info em tempo real (horários, eventos)       |
|   - Plano de manhã / tarde / noite               |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    | Entrada do Usuário|
    |------------------|
    |  Humor           |
    |  Interesses      |
    |  Orçamento       |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|           Saída: Roteiro em Markdown             |
|--------------------------------------------------|
| - Blocos de horário (manhã / tarde / noite)      |
| - Nomes dos lugares com links e horários         |
| - Atividades compatíveis com o orçamento         |
+--------------------------------------------------+
```


In [ ]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model=MODEL_NAME, generate_content_config=RETRY_CONFIG,
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
    Always reply in the same language as the user.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

In [ ]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [ ]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Planeje um passeio de um dia relaxante e com foco em arte perto de São Paulo. Tudo isso sem gastar muito!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

---
## Parte 2: Turbinando Agentes com Ferramentas Customizadas 🛠️

Até aqui usamos a poderosa ferramenta embutida `GoogleSearch`. Mas o verdadeiro poder dos agentes vem de conectá-los à **sua própria lógica e fontes de dados**.

É aí que entram as **ferramentas customizadas**. Vamos explorar três padrões para dar novas habilidades ao seu agente, com exemplos práticos do mundo real.

### 2.1 A `FunctionTool` Simples: Chamando uma API de Clima em Tempo Real

A forma mais direta de criar uma ferramenta é escrever uma função Python. É perfeito para tarefas síncronas, como buscar dados em uma API.

**Conceito-chave:** a **docstring** da função é crítica. O ADK a usa como descrição oficial da ferramenta - é ela que o LLM lê para entender o propósito, os parâmetros e quando usá-la.

Neste exemplo, criamos uma ferramenta que chama a **API gratuita e pública Open-Meteo** para obter a previsão em tempo real de **locais marcantes de São Paulo, SP** 🇧🇷. Não precisa de API key!

In [ ]:
# --- Tool Definition: a function that calls a live public API ---
import requests
import unicodedata

# Coordinates of famous landmarks in São Paulo, SP, Brazil 🇧🇷
# (avoids needing a separate geocoding API for this example)
SAO_PAULO_LANDMARKS = {
    "parque ibirapuera": (-23.5874, -46.6576),
    "masp": (-23.5614, -46.6558),
    "mercado municipal": (-23.5416, -46.6294),
    "pinacoteca": (-23.5340, -46.6336),
    "theatro municipal": (-23.5453, -46.6386),
}

# WMO weather codes used by Open-Meteo -> description in Portuguese
WEATHER_CONDITIONS = {
    0: "céu limpo", 1: "predominantemente limpo", 2: "parcialmente nublado", 3: "nublado",
    45: "nevoeiro", 48: "nevoeiro com geada", 51: "garoa leve", 53: "garoa", 55: "garoa intensa",
    61: "chuva fraca", 63: "chuva", 65: "chuva forte", 80: "pancadas de chuva leves",
    81: "pancadas de chuva", 82: "pancadas de chuva fortes", 95: "tempestade",
    96: "tempestade com granizo leve", 99: "tempestade com granizo forte",
}

def _strip_accents(text: str) -> str:
    """Removes accents to simplify matching (e.g., 'Botânico' -> 'botanico')."""
    return unicodedata.normalize("NFKD", text.lower()).encode("ascii", "ignore").decode()

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather for a famous landmark in São Paulo, Brazil.

    Args:
        location: The landmark name, e.g., "Parque Ibirapuera" or "MASP".

    Returns:
        A dictionary with temperature, feels-like, humidity, wind and sky condition (in Portuguese).
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the requested landmark
    normalized_location = _strip_accents(location)
    coords = None
    for name, latlon in SAO_PAULO_LANDMARKS.items():
        if name in normalized_location or normalized_location in name:
            coords = latlon
            break
    if not coords:
        known = ", ".join(SAO_PAULO_LANDMARKS)
        return {"status": "error",
                "message": f"Não tenho coordenadas para '{location}'. Locais conhecidos: {known}."}

    try:
        # Open-Meteo: global, free API with no key required 🌍
        # (the api.weather.gov used in the original version only covers the US)
        lat, lon = coords
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lat,
                "longitude": lon,
                "current": "temperature_2m,apparent_temperature,relative_humidity_2m,"
                           "precipitation,weather_code,wind_speed_10m",
                "timezone": "America/Sao_Paulo",
            },
            timeout=15,
        )
        response.raise_for_status()  # raise an exception for bad status codes

        current = response.json()["current"]
        return {
            "status": "success",
            "temperatura": f"{current['temperature_2m']}°C",
            "sensacao_termica": f"{current['apparent_temperature']}°C",
            "umidade": f"{current['relative_humidity_2m']}%",
            "precipitacao": f"{current['precipitation']} mm",
            "vento": f"{current['wind_speed_10m']} km/h",
            "condicao": WEATHER_CONDITIONS.get(current["weather_code"], "condição desconhecida"),
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"A chamada à API falhou: {e}"}

# --- Agent Definition: an agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model=MODEL_NAME, generate_content_config=RETRY_CONFIG,
    description="A trip planner for São Paulo that checks the real-time weather before making suggestions.",
    instruction="""You are a cautious trip planner for São Paulo, Brazil. Before suggesting any
    outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions
    at the landmark the user mentions. Incorporate the live weather details into your
    recommendation and reply in the same language as the user.""",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

In [ ]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "Quero passear ao ar livre no Parque Ibirapuera hoje, como está o tempo por lá?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

## 2.2 O Agente-como-Ferramenta: Consultando um Especialista 🧑‍🍳

Por que construir um agente que faz tudo, se você pode montar um **time de agentes especialistas**? O padrão **Agent-as-a-Tool** permite que um agente delegue uma tarefa a outro agente.

**Conceito-chave:** isso é diferente de um sub-agente. Quando o Agente A chama o Agente B como ferramenta, a resposta do Agente B **volta para o Agente A**. O Agente A então usa essa informação para compor a própria resposta final ao usuário. É uma forma poderosa de construir comportamentos complexos a partir de agentes simples, focados e reutilizáveis.

### Como Funciona

Nosso agente de nível superior, o `trip_data_concierge_agent`, atua como **Orquestrador**. Ele tem duas ferramentas à disposição:

1.  `call_db_agent`: uma função que internamente chama o nosso `db_agent` para buscar dados brutos.
2.  `call_concierge_agent`: uma função que chama o `concierge_agent`.

O próprio `concierge_agent` tem uma ferramenta: o `food_critic_agent`.

O fluxo de uma consulta complexa é:

1.  O **usuário** pede ao `trip_data_concierge_agent` um hotel e um restaurante próximo.
2.  O **Orquestrador** primeiro chama `call_db_agent` para obter os dados dos hotéis.
3.  Os dados são salvos em `tool_context.state`.
4.  O **Orquestrador** então chama `call_concierge_agent`, que recupera os dados do hotel a partir do contexto.
5.  O `concierge_agent` recebe o pedido e decide que precisa usar a própria ferramenta, o `food_critic_agent`.
6.  O `food_critic_agent` dá uma recomendação espirituosa.
7.  O `concierge_agent` recebe a resposta do crítico e a formata educadamente.
8.  Essa resposta final e polida retorna ao **Orquestrador**, que a apresenta ao usuário.

```
                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Modelo: gemini-3.1-flash-lite                            |
                         |  Descrição:                                               |
                         |   Orquestra consulta ao banco de dados e recomendação    |
                         |   de viagem                                               |
                         |-----------------------------------------------------------|
                         |  🔧 Ferramentas:                                          |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |          🔧 Ferramenta: call_db_agent     |    |      🔧 Ferramenta: call_concierge_agent     |
        |-------------------------------------------|    |---------------------------------------------|
        | Chama: db_agent                           |    | Chama: concierge_agent                       |
        |                                           |    | Usa os dados do db_agent nas recomendações  |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Modelo: gemini-3.1-flash-lite              |   | Modelo: gemini-3.1-flash-lite                   |
       | Papel: retorna dados mock (JSON) de hotéis |   | Papel: equipe do hotel que atende o usuário     |
       |        de São Paulo                         |   | Ferramentas:                                    |
       +--------------------------------------------+   |  - food_critic_agent                            |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Modelo: gemini-3.1-flash-lite                   |
                                                       | Papel: dá uma recomendação espirituosa de       |
                                                       |        restaurante em São Paulo                  |
                                                       +------------------------------------------------+
```


In [ ]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model=MODEL_NAME, generate_content_config=RETRY_CONFIG,
    instruction="You are a database agent with data about hotels in São Paulo, Brazil. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'Rosewood São Paulo', 'rating': 5, 'reviews': 450}, {'name': 'Renaissance São Paulo', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model=MODEL_NAME, generate_content_config=RETRY_CONFIG,
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location in São Paulo, Brazil. Reply in the same language as the user.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model=MODEL_NAME, generate_content_config=RETRY_CONFIG,
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely, replying in the same language as the user.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model=MODEL_NAME, generate_content_config=RETRY_CONFIG,
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.

    3.  **Always reply in the same language as the user.**
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

In [ ]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Busque no banco de dados os hotéis mais bem avaliados de São Paulo e depois sugira um lugar para jantar perto do hotel com mais avaliações."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

---
## Parte 3: Agente com Memória - O Planejador Adaptável 🗺️

Agora vamos ver um agente que não só **lembra**, como também **se adapta**. Vamos desafiar o `multi_day_trip_agent` a replanejar parte do roteiro com base no nosso feedback. Este é um teste muito mais realista de IA conversacional.

```
+-----------------------------------------------------+
|      Agente Adaptável de Viagem Multi-Dia 🗺️       |
|-----------------------------------------------------|
|  Modelo: gemini-3.1-flash-lite                      |
|  Descrição:                                         |
|   Monta roteiros de vários dias passo a passo,      |
|   lembra dos dias anteriores e se adapta ao feedback|
|-----------------------------------------------------|
|  🔧 Ferramentas:                                    |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capacidades:                                    |
|   - Memória da conversa e das preferências          |
|   - Planejamento progressivo (1 dia por vez)        |
|   - Adapta-se ao feedback do usuário                |
|   - Garante variedade de atividades entre os dias   |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |   Interação do Usuário    |
    |---------------------------|
    | - Destino                 |
    | - Duração da viagem       |
    | - Interesses & feedback   |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|         Geração do Roteiro Dia a Dia                |
|-----------------------------------------------------|
|  🗓️ Saída do Dia N (formato Markdown):              |
|   - Atividades de manhã / tarde / noite             |
|   - Personalizada e ciente do contexto              |
|   - Mudanças aceitas, feedback reconhecido          |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Planejamento do Próximo Dia 🚀               |
|-----------------------------------------------------|
| - Constrói sobre os dias anteriores                 |
| - Evita repetição                                   |
| - Pede confirmação ao usuário antes de seguir       |
+-----------------------------------------------------+
```


In [ ]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model=MODEL_NAME, generate_content_config=RETRY_CONFIG,
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        6.  **Language**: Always reply in the same language as the user.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

### Cenário 3a: Agente COM Memória (Usando UMA ÚNICA Sessão) ✅

Primeiro, o jeito certo de fazer. Usaremos **exatamente o mesmo objeto `trip_session`** durante toda a conversa. Observe como o agente lembra do contexto do Turno 1 para responder corretamente aos pedidos dos Turnos 2 e 3.

In [ ]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENTE QUE SE ADAPTA (MESMA SESSÃO) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Sessão única criada para a nossa viagem: {trip_session.id}")

    # --- Turn 1: the user starts the trip ---
    query1 = "Olá! Quero planejar um roteiro de 2 dias em São Paulo. Me interesso por lugares históricos e boa comida local."
    print(f"\n🗣️ Usuário (Turno 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: the user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "Gostei do plano, mas não sou muito fã de museus. Pode trocar a atividade da manhã do Dia 1 por outra opção histórica?"
    print(f"\n🗣️ Usuário (Turno 2 - Feedback): '{query2}'")
    print(f"⏱️ Pausa de {PAUSE_BETWEEN_QUERIES}s para respeitar o RPM...")
    await asyncio.sleep(PAUSE_BETWEEN_QUERIES)
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: the user confirms and asks to continue ---
    query3 = "Sim, o novo plano para o Dia 1 ficou perfeito! Agora planeje o Dia 2, mantendo o foco na boa comida - ouvi falar muito do Mercadão Municipal!"
    print(f"\n🗣️ Usuário (Turno 3 - Confirmação): '{query3}'")
    print(f"⏱️ Pausa de {PAUSE_BETWEEN_QUERIES}s para respeitar o RPM...")
    await asyncio.sleep(PAUSE_BETWEEN_QUERIES)
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### Cenário 3b: Agente SEM Memória (Usando Sessões SEPARADAS) ❌

Agora vejamos o que acontece se errarmos na gestão de sessões. Aqui, vamos dar ao agente um caso de amnésia criando uma **sessão nova e separada para cada turno**.

Preste atenção na resposta do agente à segunda pergunta. Como ela está em uma sessão nova, ele não tem nenhuma memória do roteiro em São Paulo que acabamos de discutir!

In [ ]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENTE COM AMNÉSIA (SESSÕES SEPARADAS) ###")
    print("#"*60)

    # --- Turn 1: the user starts the trip in the FIRST session ---
    query1 = "Olá! Quero planejar um roteiro de 2 dias em São Paulo. Me interesso por lugares históricos e boa comida local."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nSessão criada para o Turno 1: {session_one.id}")
    print(f"🗣️ Usuário (Turno 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: the user asks to continue... but in a completely NEW session ---
    query2 = "Sim, ficou perfeito! Agora planeje o Dia 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nSessão NOVINHA criada para o Turno 2: {session_two.id}")
    print(f"🗣️ Usuário (Turno 2): '{query2}'")
    print(f"⏱️ Pausa de {PAUSE_BETWEEN_QUERIES}s para respeitar o RPM...")
    await asyncio.sleep(PAUSE_BETWEEN_QUERIES)
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()

Viu? O agente ficou confuso! Provavelmente perguntou de qual destino ou de qual viagem estávamos falando. Como a segunda pergunta estava em uma sessão nova e isolada, o agente não tinha nenhuma memória de ter planejado o Dia 1 em São Paulo.

Isso ilustra perfeitamente por que **gerenciar sessões é a chave para construir agentes verdadeiramente conversacionais!**

---
## 🎉 Parabéns! 🎉

Parabéns por completar a sua aventura ADK por Ferramentas e Memória! Você deu um salto enorme: de agentes de tiro único para sistemas de IA dinâmicos e com estado.

Vamos recapitular os conceitos poderosos que você dominou:

- **Agente Fundamental & Ferramentas**: você começou construindo um "Gênio do Passeio" e o equipou com a primeira ferramenta, o GoogleSearch.

- **Ferramentas Customizadas**: você deu um novo sentido ao seu agente criando uma ferramenta que busca dados ao vivo na API Open-Meteo - com o clima de locais marcantes de São Paulo.

- **Agente-como-Ferramenta**: você orquestrou uma hierarquia sofisticada em que agentes delegam tarefas a outros agentes mais especializados, formando um time colaborativo.

- **O Poder da Memória**: o mais importante - você viu na prática como manter uma única Session persistente permite ao agente lembrar do contexto, adaptar-se ao feedback e conduzir uma conversa significativa de vários turnos.

```
                 🖖  Vida longa e próspera!

                           .ss$$$$$$$$$$$$$$$ss.
                       .s$$$$$$$$$$$$$$$$$$$$$$$$$$ss.
                   .s$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$s
                 .$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$.
               .$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$s
             .$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$s.
            $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
          .$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
          $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
        .$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
        $$$$$$$$$$$$$$$$$$$$                                  $$
        $$$$$$$$$$$$$$$$$$     .
       s$$$$$$$$$$$$$$$$$     sssss.                           s
      ss $$$$$$$$$$$$$$$          ssss                      ss$s
     sss   $$$$$$$$$$$            ssss$$$$ss              $$$$s.
    sssss   $$$$$$$$   s.      .sss$$$$$$$$sss      sss$$$$$$s
    ss$ssss  $$$$$$$  sss.  .ss$$$$$$ss$ss     .         s$$$$s.
    ss$s   s  $$$$$$ ss.ss           ..        ss           .sss
     ss   s $s $$$$$ ss..s.                   ss
      s ss$$$$s $$$$ ssssss.                 .ss
    ss     s$$$s $$$ ssssssssss..           ssss.              .
    ss      $$  s $$ ss..ss$$$$sss        .sssss              ..
     s      s$$ss  $ ssss.sss$$$sss       .sssss.     ss..
      s.     $     $ .sss . .ss$sss        .  sssssss...   ss$
        ss   sssss$$ ..ss   ...sssss            .sss.       s
          ss    $$ $ .ss..s .sssssss
            ss$$  $$  .ssssss .sssss              ..       $
                $$$$  .ss$$ssss...s.s        ..sss$$ssss.
               $$.$$$  .ss$$sss.. sss.    .ssss$$s..s$ss. s
               ss ssss  .sssssss...sss.      .ssss$$ss..  .
               ss  ssss   .ssssssss.sss.        .sss.
               ss    .ssss    .sssssssss.               s
               ss      .sssss.  .sssssss.s .. ..ss$s. .s
              ss         sssssss.  .sssssssssssss$$s. .ss
            $ss           ssssssss.   ..sssssssssss  .ss.
          $$$$.            .sssssssss.     ssssss ssss.$s
      s$$$$$$$$s.             ..ssssssssss......ssss. s$$$s
    .s$$$$$$$$$$$s.              .ssssssssssssssss. s$$$$$ss.
       s$$$$$$$$$$$$$$ss..          .ssssssssss. .s$$$$$$$$s
         s$$$$$$$$$$$$$$$$$$$ssss...   ...ss.. s$$$$$$$$$s
           s$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$s
              .s$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$s
                    .s$$$$$$$$$$$$$$$$$$$$$$$$$$$$$s
                          .s$$$$$$$$$$$$$$$$$$$$s
                                ..s$$$$$$$$$$s

        "Fascinante... agentes de IA construídos em São Paulo."
                                                     - Spock
```
